In [ ]:
!pip install transformers datasets scikit-learn openai accelerate

In [2]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from datasets import Dataset
from scipy.special import softmax

/Users/nili/PycharmProjects/exist2025/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
train_df = pd.read_csv("/Users/nili/PycharmProjects/MasterProject/exist2025/Dataset/train.csv")
dev_df = pd.read_csv("/Users/nili/PycharmProjects/MasterProject/exist2025/Dataset/val.csv")
test_df = pd.read_csv("/Users/nili/PycharmProjects/MasterProject/exist2025/Dataset/test.csv")

print(train_df.head())
print(train_df.shape)

   test_case      id language  \
0  EXIST2023  101705       es   
1  EXIST2021   11108       es   
2  EXIST2021   10405       es   
3  EXIST2021    8658       en   
4  EXIST2023  200886       en   

                                                text  \
0  segundo: el mundo, y ecuador en especial, es u...   
1  @vox_es dicen que las mujeres son valientes pe...   
2  @elisacarrio jubilate, gorda termotanque de ca...   
3  you guys don’t understand the struggle of have...   
4  boris johnson never brushing his hair shows ge...   

                                        task1  \
0  ['YES', 'YES', 'YES', 'YES', 'YES', 'YES']   
1                                      sexist   
2                                      sexist   
3                                  non-sexist   
4     ['YES', 'YES', 'NO', 'NO', 'NO', 'YES']   

                                               task2  hard_label_task1  \
0  [['MISOGYNY-NON-SEXUAL-VIOLENCE'], ['STEREOTYP...                 1   
1                    

In [4]:
import re

def preprocess(text):

    text = re.sub(r'@\w+', '@user', text)
    text = re.sub(r'http\S+', 'http', text)

    return text

train_df["tweet"] = train_df["tweet"].apply(preprocess)
dev_df["tweet"] = dev_df["tweet"].apply(preprocess)
test_df["tweet"] = test_df["tweet"].apply(preprocess)

KeyError: 'tweet'

In [5]:
#Preprocessing

In [6]:
df = pd.concat([train_df, dev_df])

train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["hard_label_task1"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

In [7]:
def convert_to_dataset(df):

    df = df.copy()

    df["text"] = df["text"].fillna("").astype(str)
    df["hard_label_task1"] = df["hard_label_task1"].astype(int)

    data = {
        "text": df["text"].tolist(),
        "label": df["hard_label_task1"].tolist()
    }

    return Dataset.from_dict(data)

train_dataset = convert_to_dataset(train_df)
val_dataset = convert_to_dataset(val_df)

In [8]:
#For test
test_dataset = Dataset.from_dict({
    "text": test_df["text"].tolist()
})

In [9]:
from transformers import AutoTokenizer
MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast= False)


In [10]:
MAX_LENGTH = 128

def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(tokenize)
val_dataset = val_dataset.map(tokenize)
test_dataset = test_dataset.map(tokenize)

Map:   0%|          | 0/13972 [00:00<?, ? examples/s]

Map:   0%|          | 0/2466 [00:00<?, ? examples/s]

Map:   0%|          | 0/1827 [00:00<?, ? examples/s]

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2 #num_labels = 6 for multiclass

)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_weighted": f1_score(labels, preds, average="weighted")
    }

In [15]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    tokenizer=tokenizer,

    compute_metrics=compute_metrics
)

/var/folders/w2/7wgmb9zn01d8k6wv97czc7th0000gn/T/ipykernel_70905/434736182.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [14]:
training_args = TrainingArguments(

    output_dir="./model",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=4,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    logging_dir="./logs",

    report_to=None
)

In [27]:
#!pip install -U accelerate


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [16]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/Users/nili/PycharmProjects/exist2025/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.588700,0.503998,0.784672,0.781371,0.783658
2,0.429300,0.495985,0.780211,0.779560,0.778540
3,0.337100,0.488012,0.801298,0.801119,0.801627
4,0.287800,0.549957,0.802514,0.801810,0.802816


/Users/nili/PycharmProjects/exist2025/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/nili/PycharmProjects/exist2025/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/nili/PycharmProjects/exist2025/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=3496, training_loss=0.40000921201378586, metrics={'train_runtime': 1983.8545, 'train_samples_per_second': 28.171, 'train_steps_per_second': 1.762, 'total_flos': 3676253593657344.0, 'train_loss': 0.40000921201378586, 'epoch': 4.0})

In [17]:
results = trainer.evaluate()

print(results)

/Users/nili/PycharmProjects/exist2025/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.48801180720329285, 'eval_accuracy': 0.8012976480129764, 'eval_f1_macro': 0.8011185579227346, 'eval_f1_weighted': 0.8016267865572049, 'eval_runtime': 29.2201, 'eval_samples_per_second': 84.394, 'eval_steps_per_second': 5.305, 'epoch': 4.0}


In [18]:
pred = trainer.predict(val_dataset)

logits = pred.predictions
labels = pred.label_ids

probs = softmax(logits, axis=1)

predictions = np.argmax(probs, axis=1)
confidence = np.max(probs, axis=1)

print(classification_report(labels, predictions))

/Users/nili/PycharmProjects/exist2025/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


              precision    recall  f1-score   support

           0       0.85      0.77      0.81      1338
           1       0.75      0.84      0.80      1128

    accuracy                           0.80      2466
   macro avg       0.80      0.80      0.80      2466
weighted avg       0.81      0.80      0.80      2466



In [21]:
from openai import OpenAI

client = OpenAI(api_key="xxx")

def query_llm(text):
    prompt = f"""You are an expert system detecting sexist content in social media posts.
Classify the tweet into one of these two categories:
0 = non-sexist
1 = sexist

### Examples:
Tweet: "I think everyone should have equal access to education regardless of gender."
Label: 0

Tweet: "Women are just naturally not as good at technical jobs as men are."
Label: 1

Tweet: "Just finished a great workout at the gym, feeling exhausted!"
Label: 0

Tweet: "She only got the promotion because of her looks, typical for her kind."
Label: 1

### Task:
Tweet: "{text}"
Return only the digit (0 or 1)."""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0, # Keeps the output consistent
            max_tokens=1
        )
        # Clean the output to ensure it's just a number
        result = response.choices[0].message.content.strip()
        return int(result)
    except Exception as e:
        print(f"Error querying LLM: {e}")
        return 0 # Default fallback

In [23]:
THRESHOLD = 0.65

texts = val_df["text"].tolist()

final_predictions = []


for i in range(len(predictions)):

    if confidence[i] >= THRESHOLD:

        final_predictions.append(predictions[i])

    else:

        llm_pred = query_llm(texts[i])

        final_predictions.append(llm_pred)

In [24]:
print(
    classification_report(
        labels,
        final_predictions
    )
)

              precision    recall  f1-score   support

           0       0.86      0.77      0.81      1338
           1       0.76      0.86      0.80      1128

    accuracy                           0.81      2466
   macro avg       0.81      0.81      0.81      2466
weighted avg       0.82      0.81      0.81      2466



In [25]:
test_pred = trainer.predict(test_dataset)

test_logits = test_pred.predictions

test_probs = softmax(test_logits, axis=1)

test_preds = np.argmax(test_probs, axis=1)
test_conf = np.max(test_probs, axis=1)

/Users/nili/PycharmProjects/exist2025/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [26]:
test_texts = test_df["text"].tolist()

final_test_predictions = []

for i in range(len(test_preds)):

    if test_conf[i] >= THRESHOLD:

        final_test_predictions.append(test_preds[i])

    else:

        llm_pred = query_llm(test_texts[i])

        final_test_predictions.append(llm_pred)

In [27]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "label": final_test_predictions
})

submission.to_csv("submission.csv", index=False)